# Helpdesk API — seed de tickets para el demo

Crea tickets de prueba variados contra la API desplegada en EC2, como insumo
para el analisis en vivo (que se escribe durante la sesion, no aqui).

Antes de correr, define `HELPDESK_BASE_URL` con la IP publica de la instancia, por ejemplo:

```bash
export HELPDESK_BASE_URL=http://203.0.113.10:8000
```

In [ ]:
import os

import requests

BASE_URL = os.environ.get("HELPDESK_BASE_URL", "http://localhost:8000")
print(f"Usando BASE_URL={BASE_URL}")

In [ ]:
SAMPLE_TICKETS = [
    {"title": "No puedo iniciar sesion en POS", "description": "El terminal de la tienda 12 no acepta el usuario del cajero.", "priority": "high", "requester_email": "tienda12@kindor.co"},
    {"title": "Impresora de recibos sin papel", "description": "La impresora fiscal de la caja 3 no imprime tickets desde ayer.", "priority": "medium", "requester_email": "tienda07@kindor.co"},
    {"title": "Lector de codigo de barras roto", "description": "El scanner inalambrico dejo de leer codigos en la caja 1.", "priority": "medium", "requester_email": "tienda03@kindor.co"},
    {"title": "Internet caido en sucursal", "description": "La sucursal centro reporta sin conexion desde las 9am.", "priority": "critical", "requester_email": "tienda01@kindor.co"},
    {"title": "Actualizacion de precios pendiente", "description": "Los precios de la promocion de fin de semana no se reflejan en el sistema.", "priority": "low", "requester_email": "tienda05@kindor.co"},
    {"title": "Terminal de pago rechaza tarjetas", "description": "El datafono de la caja 2 rechaza todas las tarjetas de credito.", "priority": "critical", "requester_email": "tienda09@kindor.co"},
    {"title": "Reporte diario no llega por correo", "description": "El reporte de cierre de caja no se envio esta manana.", "priority": "low", "requester_email": "tienda02@kindor.co"},
    {"title": "Bascula de piso descalibrada", "description": "La bascula del area de frutas marca pesos incorrectos.", "priority": "medium", "requester_email": "tienda11@kindor.co"},
    {"title": "Camara de seguridad sin senal", "description": "La camara de la entrada principal no transmite video.", "priority": "high", "requester_email": "tienda06@kindor.co"},
    {"title": "Sistema lento al abrir tickets", "description": "Abrir un nuevo ticket de soporte tarda mas de un minuto.", "priority": "medium", "requester_email": "tienda14@kindor.co"},
]

created = []
for payload in SAMPLE_TICKETS:
    response = requests.post(f"{BASE_URL}/tickets", json=payload, timeout=10)
    response.raise_for_status()
    created.append(response.json())

print(f"Creados {len(created)} tickets")

In [ ]:
# Todos nacen en 'open'; movemos algunos a otros estados validos para
# que la grafica final no sea una sola barra.
STATUS_UPDATES = ["in_progress", "in_progress", "pending", "closed", "closed"]

for ticket, new_status in zip(created, STATUS_UPDATES):
    response = requests.patch(
        f"{BASE_URL}/tickets/{ticket['id']}",
        json={"status": new_status},
        timeout=10,
    )
    response.raise_for_status()

print("Estados actualizados")

## Analisis en vivo

`created` tiene los tickets recien creados (con su `id`, `status`, `priority`).
El endpoint `GET /tickets/stats/summary` da conteos por estado y prioridad;
`GET /tickets/stats/workload` da carga por agente. Las celdas de analisis
(que consultan uno de estos y grafican con matplotlib) se escriben aqui
durante la sesion en vivo, no antes.